# Semenov et al. (2026) on dandiset 001518

**Paper.** Semenov I, Silkuniene G, Silkunas M, Pakhomova ON, Bixler JN, Ibey BL, Kochunov P,
Pakhomov AG (2026). Sub-microsecond optical measurements of cell membrane charging and lesioning
by pulsed electric fields. Bioelectrochemistry 171, 109283. doi:10.1016/j.bioelechem.2026.109283.

**Dataset.** DANDI:001518, version 0.261006.1957
(doi:10.48324/dandi.001518/0.261006.1957), streamed from the archive. The paper cites no
version of the dataset. This is the only published version and the most recent one, so it
is pinned (protocol, author step 1.3). The draft has the same 190 assets.

**Claims.** `claims/001518/10.1016-j.bioelechem.2026.109283.yaml`, written from the version of
record (main text, Methods, Results, Discussion and figure legends; no supplement).

**Outside input.** The charging time constant of Eq. (1) is taken from reference [20] of the
paper, Semenov et al. (2025), Bioelectrochemistry 168, 109143,
doi:10.1016/j.bioelechem.2025.109143: tau = 44.5 ns/um x R, the fit through the origin in its
Results. The paper says it used this fit (Methods 2.3).

Claims and how each is treated (the outcomes are those of the attempt record):

| Claim | Tested | Outcome |
|---|---|---|
| fig3-dye-sensitivity | tested | partially reproduced |
| fig3-fluovolt-nonlinear | tested | not reproduced |
| fig2a-fluovolt-saturation | tested | not reproduced |
| fig2a-fluovolt-stable | tested | not reproduced |
| fig4a-fluovolt-limits | tested | not reproduced |
| fig4a-fluovolt-cathodic-decline | tested | not reproduced |
| fig5b-conductivity-calibration | tested (Eq. 2, no recordings) | partially reproduced |
| fig1-pole-dimming | run under the top mapping only | blocked: data missing (no sets above 430 V/cm under the adopted mapping) |
| fig2b-di8-climbs-to-570 | as above | blocked: data missing |
| fig2b-di8-decline-within-set | as above | blocked: data missing |
| fig4b-di8-limits | as above | blocked: data missing |
| fig4b-first-10-pulses | as above | blocked: data missing |
| fig4b-cathodic-asymmetry | as above | blocked: data missing |
| fig4c-equator-follows-cosine | as above | blocked: data missing |
| fig7b-charging-during-pulse | as above | blocked: data missing |
| fig2cd-stable-before-electroporation, fig2c-1000-within-set, fig2d-lasting-reduction | not in the dataset (alternating protocol) | blocked: data missing |
| fig4d-short-pulses-same-limit | not in the dataset (0.3-us pulses) | blocked: data missing |
| fig5a-gd-extends-anodic, fig5a-gd-linear, fig5a-gd-cathodic, fig5a-gd-differences, fig5c-gd-reduces-conductivity | not in the dataset (Gd3+) | blocked: data missing |
| fig6a-ca-single-pulse, fig6b-ca-train-responders, fig6b-ca-650 | not in the dataset (Fluo-4 Ca2+ imaging) | blocked: data missing |
| fig7ab-test-pulse-recovery, fig7c-fast-conductivity-drop, fig7c-full-reseal-mild | not in the dataset (two-pulse protocol) | blocked: data missing |

Every tested outcome except fig5b-conductivity-calibration rests on assumptions: the factor from volts to V/cm (the `bottom`
mapping, data check), the pixel size, and which of the cells in each field of view stand for
the paper's one cell per experiment.

In [ ]:
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.nonparametric.smoothers_lowess import lowess

import pef

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("dandiset", pef.DANDISET, "version", pef.VERSION)

In [ ]:
# Streams all 190 files (about 7 GB) and keeps only the membrane profiles; about 15 minutes.
started = time.time()
sets = pef.load_all()

In [ ]:
tracked = pef.track(sets)
resp = pef.responding(tracked)
D = pef.pole_rows(resp)  # background subtracted, reading (b)
D_raw = pef.pole_rows(resp, background=False)  # no background subtraction, reading (a)
print(f"{len(sets)} sets loaded and reduced in {(time.time() - started) / 60:.1f} min")

## Data check

Each file is one set of 100 images. The session description gives the dye and the pulse
amplitude in volts; there is no table of trials, no cell outline, no field direction and no
pixel size. The checks below compare what the paper describes with the values.

In [ ]:
inv = pd.DataFrame([dict(subject=s["subject"], dye=s["dye"], volt=s["volt"], start=s["start"],
                         spacing_mm=s["spacing_mm"], n_frames=len(s["good"]), n_dark=int((~s["good"]).sum()),
                         dark_first=bool(~s["good"][0]), n_cells_found=len(s["cells"])) for s in sets])
print("Sets:", len(inv), " subjects:", inv.subject.nunique(), " frames per set:", sorted(inv.n_frames.unique()))
print("\nSets per subject and dye:")
print(inv.groupby(["dye", "subject"]).agg(sets=("volt", "size"), volts=("volt", lambda v: f"{v.min()}-{v.max()} V"),
                                           span_min=("start", lambda t: round((t.max() - t.min()) / 60, 1))))
print("\nVoltages (same 19 in every subject):", sorted(inv.volt.unique()))
print("Every subject has each voltage once:", bool((inv.groupby(["subject", "volt"]).size() == 1).all()))
order = inv.sort_values("start").groupby("subject").volt.apply(lambda v: bool(np.all(np.diff(v.values) > 0)))
print("Voltage increases set by set in every subject:", bool(order.all()))
gaps = inv.sort_values("start").groupby("subject").start.apply(lambda t: np.diff(t.values)).explode().astype(float)
print(f"Time between set starts: median {gaps.median():.0f} s, range {gaps.min():.0f} to {gaps.max():.0f} s")
print("Electrode distance stated in the files (mm):", sorted(inv.spacing_mm.unique()), "; paper, Methods 2.2: 1.6 mm")

The dandiset holds ten experiments from one day, five with Di-8-ANEPPS (c1 to c5) and five with
FluoVolt (c7 to c11). Each is one escalation of 19 sets of 1-us pulses from 25 to 250 V, with the
voltage raised set by set. The paper's escalations (Fig. 2A, 2B and 4A, 4B) have nine sets, 43 to
1400 V/cm (Fig. 2 legend: "9 cycles"), with n = 4 for FluoVolt and 5 for Di-8-ANEPPS. The files
say the electrodes are 1.00 mm apart; the paper says 1.6 mm. Nothing in the files records the
alternating protocol (Fig. 2C, D), 0.3-us pulses (Fig. 4D), Gd3+ (Fig. 5), Fluo-4 Ca2+ imaging
(Fig. 6) or the two-pulse protocol (Fig. 7).

The voltages span a factor of 10, while the paper's field strengths span a factor of 1400 / 43 =
33. Field strength is proportional to voltage (Methods 2.2), so no single factor maps all nine
of the paper's levels onto these sets. The factor from volts to V/cm is in neither the paper,
its reference [20], nor the files. Two readings are run: `top` puts the paper's highest level
on the highest voltage (1400 V/cm at 250 V, 5.6 V/cm per V), and `bottom` puts its lowest on the
lowest (43 V/cm at 25 V, 1.72 V/cm per V). The check of the dye response below decides between them.

In [ ]:
print("Dark frames (laser misfires): frame 0 is dark in", int(inv.dark_first.sum()), "of", len(inv), "sets;",
      int((inv.n_dark - inv.dark_first).sum()), "other dark frames in", int(((inv.n_dark - inv.dark_first) > 0).sum()), "sets")
fm = np.array([s["frame_mean"] for s in sets])
good = np.array([s["good"] for s in sets])
print(f"Whole-image mean: lit frames {np.median(fm[good]):.0f} (range {fm[good].min():.0f} to {fm[good].max():.0f}),"
      f" dark frames {np.median(fm[~good]):.0f} (range {fm[~good].min():.0f} to {fm[~good].max():.0f}) counts")
bg = np.concatenate([s["bg"][s["good"]] for s in sets])
print(f"Cell-free background in lit frames: median {np.median(bg):.0f}, range {bg.min():.0f} to {bg.max():.0f} counts (unit 'n.a.' in the files)")

Frame 0 of almost every set, and a few other frames, are dark: their mean is near the camera
offset, so the laser did not fire. They are left out (helper, `good_frames`); with frame 0 out,
the baseline has 39 images in most sets, not the paper's 40. The cell-free background is
about 330 counts, a large part of the membrane signal, so background subtraction about doubles
dF (ranges printed below); both readings are run where the claims file names them.

In [ ]:
cell_tab = pd.DataFrame([dict(subject=sub, dye=ss[0]["dye"], found_first_set=len(ss[0]["cells"]), in_every_set=n_full,
                              responding=len(keep), cathode_deg=round(cath))
                         for sub, (ss, keep, cath, n_full) in sorted(resp.items())])
print(cell_tab.to_string(index=False))
print("\nResponding cells:", int(cell_tab.responding.sum()), "; by dye:", cell_tab.groupby("dye").responding.sum().to_dict())
geo = D.groupby(["dye", "subject", "cell"])[["R_area_um", "R_perim_um", "dist_centre_px"]].mean()
print(f"Cell radius (um, pixel {pef.PIXEL_UM:.4f} um): equal-area {geo.R_area_um.min():.2f} to {geo.R_area_um.max():.2f}, "
      f"mean {geo.R_area_um.mean():.2f}; perimeter/2pi {geo.R_perim_um.min():.2f} to {geo.R_perim_um.max():.2f}, mean {geo.R_perim_um.mean():.2f}")
print("Mean radius by dye (equal-area):", geo.groupby("dye").R_area_um.mean().round(2).to_dict())

Every field of view holds several cells (3 to 6 found), while the paper analysed one cell per
experiment and does not say which. The cell analysed in each experiment cannot be identified
from the files. Three readings are run: (i) the mean of the experiment's responding cells, with
the experiment as the unit (n = 5 per dye); (ii) the responding cell nearest the image centre,
one per experiment; (iii) every responding cell as a unit. The field direction, which the files
do not record, is found from the data: in every experiment the side whose emission rises (the
cathode-facing side, Methods 2.3) lies between 94 and 106 degrees of the image's +x axis, that
is towards the bottom of the image. With a 16-um camera pixel and the 60x objective, the
mean radius is about 6.7 um. The paper gives about 8 um for CHO cells as a typical value,
so the pixel size is consistent with the data but not shown by them.

In [ ]:
# Recordings deposited twice: the frame-to-frame fluctuation of the laser makes each set's
# sequence of whole-image means a fingerprint. Compare every pair of sets.
x = fm.copy()
x[~good] = np.nan
x = (x - np.nanmean(x, 1, keepdims=True)) / np.nanstd(x, 1, keepdims=True)
x = np.nan_to_num(x)
cc = x @ x.T / x.shape[1]
np.fill_diagonal(cc, np.nan)
iu = np.triu_indices(len(sets), 1)
print(f"Correlation of frame-mean sequences over {len(iu[0])} pairs of sets: max {np.nanmax(cc[iu]):.2f}, "
      f"99th percentile {np.nanpercentile(cc[iu], 99):.2f}")
identical = [(i, j) for i, j in zip(*iu) if np.array_equal(sets[i]["frame_mean"], sets[j]["frame_mean"])]
print("Pairs with identical frame means:", len(identical))
prof0 = {}
for s in sets:
    for c in s["cells"]:
        prof0.setdefault(round(float(c["prof"][1].sum()), 3), []).append(s["asset_id"])
print("Cell profiles shared by two files:", sum(len(v) > 1 for v in prof0.values()))

No two sets share their frame-by-frame laser fluctuation or a cell profile, so no recording is
deposited twice. The cells of each experiment are linked from set to set by position
(within 10 pixels); the same object is found in all 19 sets, so a cell is treated as the same
across the sets of an experiment.

In [ ]:
# Shared clock: the pulse starts at image 41 (index 40) in every set (Methods 2.3: 40 baseline
# images, then 60 with the pulse). Polarization (cathodic minus anodic dF, halved) in 10-image
# windows either side of a boundary; the step should be largest at the true onset, and absent at
# the same boundary in the lowest-voltage sets if the response is a response.
def polarization_frames(df):
    return np.stack([(a - b) / 2 for a, b in zip(df.frames_cath, df.frames_anod)])

onsets, steps = [], []
for (sub, volt), g in D.groupby(["subject", "volt"]):
    pol = np.nanmean(polarization_frames(g), 0)
    st = {b: np.nanmean(pol[b:b + 10]) - np.nanmean(pol[b - 10:b]) for b in range(15, 86)}
    onsets.append(dict(subject=sub, volt=volt, onset=max(st, key=lambda b: abs(st[b])), step40=st[40],
                       step30=st[30], step50=st[50]))
on = pd.DataFrame(onsets)
print("Boundary with the largest polarization step, sets at 50 V and above:")
print(on[on.volt >= 50].onset.value_counts().sort_index().to_string())
print("\nMean step (% dF) at boundaries 30, 40, 50, by voltage group:")
print(on.assign(group=pd.cut(on.volt, [0, 25, 60, 120, 250])).groupby("group", observed=True)[["step30", "step40", "step50"]].mean().round(2))

In every set at 50 V and above, the polarization step is largest at index 40 or within a frame
or two of it, and it grows with voltage; at boundaries 30 and 50 the step is near zero. The
images are therefore on the clock the paper describes: 40 images without the pulse, then 60
with it. Because there is one stream per file, no other stream needs aligning.

In [ ]:
# Field strength per volt implied by the low-field response under the paper's calibration
# (Fig. 3: 26 and 9.1% dF per 100 mV) and Eq. (1). Per cell, the slope of the half-difference
# between the poles (which cancels the uniform bleaching offset) over the sets at 25 to 60 V.
rows = []
for (dye, sub, cell), g in D.groupby(["dye", "subject", "cell"]):
    g = g[g.volt <= 60]
    half = (g.dF_cath - g.dF_anod) / 2
    slope = np.sum(half * g.volt) / np.sum(g.volt ** 2)
    R = g.R_area_um.mean()
    rows.append(dict(dye=dye, subject=sub, cell=cell, k=slope / (pef.SENS[dye] / 100 * pef.impc_mV(1.0, R, 0))))
kk = pd.DataFrame(rows)
print("Implied V/cm per volt, per cell:")
print(kk.groupby("dye").k.describe().round(2))
print("\nReadings: bottom", round(pef.FIELD_PER_VOLT["bottom"], 2), " top", round(pef.FIELD_PER_VOLT["top"], 2))
low = D[D.volt == 25]
print(f"\nAt 25 V, mean of the two poles (bleaching offset): Di-8-ANEPPS {((low.dF_cath + low.dF_anod) / 2)[low.dye == 'Di8'].mean():.1f}%, "
      f"FluoVolt {((low.dF_cath + low.dF_anod) / 2)[low.dye == 'FV'].mean():.1f}%")

Under the paper's dye sensitivities, the low-field response gives a median of 1.5 to 1.6 V/cm per
volt for each dye, measured independently in each. That is close to the `bottom`
reading (1.72) and far from the `top` reading (5.6): under `top`, both dyes would be 3 to 4 times
less sensitive than the paper reports. The `bottom` reading is adopted as an assumption, and
`top` is shown as a variant throughout. Under `bottom` the sets cover 43 to 430 V/cm. This is
the range of the paper's calibration (Fig. 3, 43 to 360 V/cm) and below the electroporating
fields of 570 to 1400 V/cm. The 1.00 mm electrode distance stated in the files would give a nominal
field (voltage over distance) of 10 V/cm per volt, and the paper's 1.6 mm 6.25, so neither fits. The check uses the
paper's calibration, so it supports the `bottom` reading but does not prove it. At 25 V the
mean of the two poles is -3.9% (Di-8-ANEPPS) and -2.1% (FluoVolt), a uniform offset like the
bleaching the paper reports for Di-8-ANEPPS at 43 V/cm (Results 3.2, about 5%).

In [ ]:
# Ranges of the variables used.
for col in ["dF_cath", "dF_anod", "pi_cath", "pi_anod", "f10_cath", "f10_anod"]:
    print(f"{col:9s} by dye: " + ", ".join(f"{d} {g[col].min():.1f} to {g[col].max():.1f} %" for d, g in D.groupby("dye")))
print("Without background subtraction:", ", ".join(f"{d} cathodic {g.dF_cath.min():.1f} to {g.dF_cath.max():.1f} %" for d, g in D_raw.groupby("dye")))

In [ ]:
# Pole dF against voltage, mean of experiment means (reading i), both dyes.
def units(df, reading):
    # (i) experiment mean of its cells; (ii) the cell nearest the image centre; (iii) every cell.
    if reading == "experiment":
        g = df.groupby(["dye", "subject", "volt"], as_index=False)
        out = g.mean(numeric_only=True)
        out["unit"] = out.subject
        return out
    if reading == "centre":
        pick = df.groupby("subject").apply(lambda g: g.loc[g.dist_centre_px.idxmin(), "cell"])
        out = df[df.cell == df.subject.map(pick)].copy()
        out["unit"] = out.subject
        return out
    out = df.copy()
    out["unit"] = out.subject + "/" + out.cell.astype(str)
    return out

READINGS = ["experiment", "centre", "cells"]
K = pef.FIELD_PER_VOLT
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, dye in zip(axes, ["FV", "Di8"]):
    u = units(D[D.dye == dye], "experiment")
    for pole, col in [("cathodic", "dF_cath"), ("anodic", "dF_anod")]:
        m = u.groupby("volt")[col].agg(["mean", "sem"])
        ax.errorbar(m.index, m["mean"], m["sem"], marker="o", ms=3, label=pole)
    ax.set_title({"FV": "FluoVolt", "Di8": "Di-8-ANEPPS"}[dye] + " (n = 5 experiments)")
    ax.set_xlabel("pulse amplitude (V)")
    sec = ax.secondary_xaxis("top", functions=(lambda v: v * K["bottom"], lambda e: e / K["bottom"]))
    sec.set_xlabel("V/cm, bottom reading")
    ax.axhline(0, color="0.7", lw=0.5)
axes[0].set_ylabel("collapsed pole dF (%)")
axes[0].legend()
plt.tight_layout()
plt.show()
mm = units(D, "experiment").groupby(["dye", "volt"])[["dF_cath", "dF_anod"]].mean()
print("Plotted means, range:", mm.min().round(1).to_dict(), mm.max().round(1).to_dict(), "; no point outside the axes")

## fig3-dye-sensitivity

> Fitted linearly through the origin over -150 to +150 mV IMPC, FluoVolt changes by 26% and
> Di-8-ANEPPS by 9.1% dF per 100 mV, with correlation coefficients 0.96 and 0.78.

Every perimeter point of every analysed cell, in the sets from 43 to 360 V/cm, against IMPC from
Eq. (1) with the cell's own radius and tau = 44.5 ns/um x R (reference [20]). Least squares
through the origin over IMPC from -150 to +150 mV. Readings run: field mapping (bottom, top),
background (b subtracted, a none), radius (equal-area, perimeter / 2 pi), cells (all
responding cells, or the cell nearest the image centre), and sets (every set from 43 to 360
V/cm, or only the sets nearest the paper's levels 43, 150, 230 and 360 V/cm, which is the
paper's escalation). The correlation coefficient is read as (a) Pearson r and (b) the square
root of the uncentred R2.

In [ ]:
def calib_points(dye, mapping, background, radius, cells, which_sets):
    df = (D if background else D_raw)
    df = df[df.dye == dye]
    if cells == "centre":
        df = units(df, "centre")
    k = K[mapping]
    if which_sets == "levels":
        lv = pef.nearest_sets(df.volt, k)
        df = df[df.volt.isin([v for L, v in lv.items() if L <= 360])]
    else:
        df = df[df.volt * k <= 360]
    th = np.arange(360)
    X = np.concatenate([pef.impc_mV(v * k, r, th) for v, r in zip(df.volt, df[f"R_{radius}_um"])])
    Y = np.concatenate(df.profile.values)
    return X, Y, df

def fit0(X, Y, lim=150):
    m = np.abs(X) <= lim
    x, y = X[m], Y[m]
    return dict(slope=100 * np.sum(x * y) / np.sum(x * x), r_pearson=np.corrcoef(x, y)[0, 1],
                r_uncentred=np.sum(x * y) / np.sqrt(np.sum(x * x) * np.sum(y * y)), n_points=int(m.sum()))

EXP3 = {"FV": (26, 2.6, 0.96, 0.096), "Di8": (9.1, 0.91, 0.78, 0.078)}
rows = []
for mapping in ["bottom", "top"]:
    for background in [True, False]:
        for radius in ["area", "perim"]:
            for cells in ["all", "centre"]:
                for which in ["all", "levels"]:
                    for dye in ["FV", "Di8"]:
                        X, Y, df = calib_points(dye, mapping, background, radius, cells, which)
                        f = fit0(X, Y)
                        s0, ts, r0, tr = EXP3[dye]
                        rows.append(dict(mapping=mapping, bg="b" if background else "a", radius=radius, cells=cells, sets=which,
                                         dye=dye, n_sets=df.volt.nunique(), n_cells=df.groupby(["subject", "cell"]).ngroups, **f,
                                         slope_ok=abs(f["slope"] - s0) <= ts, r_a_ok=abs(f["r_pearson"] - r0) <= tr,
                                         r_b_ok=abs(f["r_uncentred"] - r0) <= tr))
cal = pd.DataFrame(rows)
print(cal.round(3).to_string(index=False))

In [ ]:
# A reading reproduces when, for both dyes, the slope is within tolerance and r is within
# tolerance under (a) or (b).
w = cal.assign(ok=cal.slope_ok & (cal.r_a_ok | cal.r_b_ok)).pivot_table(
    index=["mapping", "bg", "radius", "cells", "sets"], columns="dye", values=["slope", "ok"], aggfunc="first")
w[("both_ok", "")] = w[("ok", "FV")].astype(bool) & w[("ok", "Di8")].astype(bool)
print(w.round(2).to_string())
print("\nReadings where both dyes meet the criterion:", int(w[("both_ok", "")].sum()), "of", len(w))
main = cal[(cal.mapping == "bottom") & (cal.bg == "b") & (cal.radius == "area") & (cal.cells == "all") & (cal.sets == "all")]
print("\nAdopted reading (bottom, background subtracted, equal-area radius, all cells, all sets):")
print(main[["dye", "slope", "r_pearson", "r_uncentred", "n_points", "n_cells", "n_sets"]].round(3).to_string(index=False))
by_exp = []
for dye in ["FV", "Di8"]:
    X, Y, df = calib_points(dye, "bottom", True, "area", "all", "all")
    n = 360
    sub = np.repeat(df.subject.values, n)
    for s in np.unique(sub):
        by_exp.append(dict(dye=dye, subject=s, **fit0(X[sub == s], Y[sub == s])))
print("\nBy experiment, adopted reading:")
print(pd.DataFrame(by_exp).round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, dye in zip(axes, ["FV", "Di8"]):
    X, Y, _ = calib_points(dye, "bottom", True, "area", "all", "all")
    ax.plot(X[::7], Y[::7], ",", alpha=0.3)
    f = fit0(X, Y)
    g = np.linspace(-450, 450, 10)
    ax.plot(g, f["slope"] / 100 * g, "k--", label=f"fit {f['slope']:.1f}%/100 mV")
    ax.plot(g, EXP3[dye][0] / 100 * g, "r:", label=f"paper {EXP3[dye][0]}%/100 mV")
    ax.set_ylim(-100, 100)
    ax.set_xlabel("IMPC (mV)")
    ax.set_title({"FV": "FluoVolt", "Di8": "Di-8-ANEPPS"}[dye])
    ax.legend(fontsize=8)
    print(dye, f"points {len(X)}, every 7th plotted; dF range {Y.min():.0f} to {Y.max():.0f} %;"
          f" outside the axes {(np.abs(Y) > 100).sum()} points; IMPC range {X.min():.0f} to {X.max():.0f} mV")
axes[0].set_ylabel("dF (%)")
plt.tight_layout()
plt.show()

**Result: partially reproduced.** Under the adopted reading (bottom mapping, background subtracted,
equal-area radius, all responding cells, all 18 sets from 43 to 344 V/cm), FluoVolt gives
23.98% dF per 100 mV (expected 26 +- 2.6) with r = 0.908 (Pearson) and 0.902 (uncentred), and
Di-8-ANEPPS gives 10.52% (expected 9.1 +- 0.91, so above the bound of 10.01) with r = 0.824 and
0.804, both within 0.78 +- 0.078. With only the cell nearest the image centre, Di-8-ANEPPS gives
9.04% and FluoVolt 22.46%, below its bound of 23.4. None of the 32 readings puts both slopes
within tolerance. Without background subtraction the slopes are 11.67 and 5.28, and under the
top mapping 6.10 and 2.37. By experiment, the slopes are 21.2 to 25.9 (FluoVolt) and 10.0 to 11.3
(Di-8-ANEPPS). The field mapping was checked against this same low-field response (data check),
so the agreement supports the bottom mapping and the paper's calibration together and cannot
test either one alone.

## fig3-fluovolt-nonlinear

> FluoVolt departed from linearity from about +-200 mV IMPC, while Di-8-ANEPPS stayed nearly
> linear from -500 to +500 mV; so FluoVolt's early departure reflects its dynamic range, not
> electroporation.

The points and line of fig3-dye-sensitivity (adopted reading; the top mapping as a variant).
Smooths: LOWESS with span (a) 0.3, (b) 0.5, (c) 0.75, and (d) means in 50-mV bins. A departure is
the smooth lying closer to zero than the line by more than 10% of the line's value. Checked on a
10-mV grid; LOWESS is evaluated with `delta` set to 1% of the IMPC range, which interpolates
between nearby fitted points (a standard speed-up that cannot change the outcome).

In [ ]:
def smooth(X, Y, how):
    o = np.argsort(X)
    X, Y = X[o], Y[o]
    if how == "bins":
        edges = np.arange(np.floor(X.min() / 50) * 50, X.max() + 50, 50)
        idx = np.digitize(X, edges)
        xs = np.array([X[idx == i].mean() for i in np.unique(idx) if (idx == i).sum() >= 20])
        ys = np.array([Y[idx == i].mean() for i in np.unique(idx) if (idx == i).sum() >= 20])
        return xs, ys
    fr = lowess(Y, X, frac=how, it=0, delta=0.01 * (X.max() - X.min()), return_sorted=True)
    return fr[:, 0], fr[:, 1]

def departure(X, Y, slope, how):
    xs, ys = smooth(X, Y, how)
    grid = np.arange(np.ceil(xs.min() / 10) * 10, xs.max(), 10)
    sm = np.interp(grid, xs, ys)
    line = slope / 100 * grid
    dep = (np.abs(line) - np.sign(line) * sm) > 0.1 * np.abs(line)
    dep[grid == 0] = False
    return grid, sm, line, dep

t0 = time.time()
res = []
curves = {}
for mapping in ["bottom", "top"]:
    for dye in ["FV", "Di8"]:
        X, Y, _ = calib_points(dye, mapping, True, "area", "all", "all")
        slope = fit0(X, Y)["slope"]
        for how in [0.3, 0.5, 0.75, "bins"]:
            grid, sm, line, dep = departure(X, Y, slope, how)
            curves[(mapping, dye, how)] = (grid, sm, line, dep)
            if dye == "FV":
                hi = np.abs(grid) >= 220
                ok = bool(dep[hi].all()) if hi.any() else False
                first = grid[dep & (grid > 0)].min() if (dep & (grid > 0)).any() else np.nan
                firstn = grid[dep & (grid < 0)].max() if (dep & (grid < 0)).any() else np.nan
                res.append(dict(mapping=mapping, dye=dye, smooth=how, data_range=f"{grid.min():.0f} to {grid.max():.0f}",
                                grid_pts_220_to_end=int(hi.sum()), departs_at_all=ok, departing_pts=int(dep[hi].sum()),
                                first_dep_pos=first, first_dep_neg=firstn))
            else:
                inr = np.abs(grid) <= 500
                res.append(dict(mapping=mapping, dye=dye, smooth=how, data_range=f"{grid.min():.0f} to {grid.max():.0f}",
                                grid_pts_in_range=int(inr.sum()), departs_anywhere=bool(dep[inr].any()),
                                departing_pts=int(dep[inr].sum()),
                                departing_at=", ".join(f"{g:.0f}" for g in grid[inr & dep][:12])))
print(pd.DataFrame(res).to_string(index=False))
print(f"({time.time() - t0:.0f} s)")

In [ ]:
# Size of the departure: smooth as % of the line at chosen IMPC values, adopted mapping.
rows = []
for dye in ["FV", "Di8"]:
    for how in [0.3, 0.5, 0.75, "bins"]:
        grid, sm, line, dep = curves[("bottom", dye, how)]
        r = {"dye": dye, "smooth": how}
        for g in [-400, -300, -220, -150, 150, 220, 300, 400]:
            if grid.min() <= g <= grid.max():
                r[g] = round(100 * np.interp(g, grid, sm) / (EXP3[dye][0] if False else 1) / np.interp(g, grid, line), 1)
        rows.append(r)
print("Smooth as % of the fitted line (100 = on the line; below 90 = departure), bottom mapping:")
print(pd.DataFrame(rows).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, dye in zip(axes, ["FV", "Di8"]):
    for how in [0.3, 0.5, 0.75, "bins"]:
        grid, sm, line, dep = curves[("bottom", dye, how)]
        ax.plot(grid, sm, label=str(how))
    ax.plot(grid, line, "k--", label="line")
    ax.set_title({"FV": "FluoVolt", "Di8": "Di-8-ANEPPS"}[dye])
    ax.set_xlabel("IMPC (mV)")
    ax.legend(fontsize=7)
axes[0].set_ylabel("smoothed dF (%)")
plt.tight_layout()
plt.show()

**Result: not reproduced.** Under the bottom mapping the FluoVolt smooth departs from the line at every
|IMPC| from 220 mV to the end of the data (+-440 mV) under all four smooths: at +400 mV it is
21 to 47% of the line, at -400 mV 45 to 68%. Di-8-ANEPPS departs as well: its smooth is 57 to
84% of the line at +400 mV and 48 to 79% at -400 mV, with 58 to 64 departing grid points under
every smooth. So Di-8-ANEPPS does not stay linear over the range the data cover (+-430 mV).
Under the top mapping FluoVolt does not depart under any LOWESS span, and Di-8-ANEPPS departs
too. The claim fails under both mappings.

## fig2a-fluovolt-saturation

> With FluoVolt, pole dF rose with field strength until it saturated at 66% (255 mV) at 360
> V/cm at the cathodic pole and -83% (-320 mV) at 570 V/cm at the anodic pole; stronger fields
> did not increase the anodic response and decreased the cathodic one.

Per-image pole dF (the Fig. 2 measure, Methods 2.3), averaged over the 60 pulse images of each
set, per cell, then across units. The paper's levels are the sets nearest them (helper,
`nearest_sets`, within 15%). Both mappings, three cell readings, and the background
readings are run.

In [ ]:
def level_table(df, mapping, cols, reading):
    lv = pef.nearest_sets(df.volt, K[mapping])
    u = units(df, reading)
    out = {}
    for L, v in lv.items():
        g = u[u.volt == v]
        out[L] = {c: (g[c].mean(), g[c].sem(), len(g)) for c in cols}
    return out, lv

def show_levels(dye, cols, background=True, readings=READINGS):
    df = (D if background else D_raw)
    df = df[df.dye == dye]
    rows = []
    for mapping in ["bottom", "top"]:
        for reading in readings:
            tab, lv = level_table(df, mapping, cols, reading)
            for L, d in tab.items():
                r = dict(mapping=mapping, cells=reading, level=L, volt=lv[L], n=d[cols[0]][2])
                for c in cols:
                    r[c] = round(d[c][0], 1)
                    r[c + "_se"] = round(d[c][1], 1)
                rows.append(r)
    return pd.DataFrame(rows)

fv2 = show_levels("FV", ["pi_cath", "pi_anod"])
print(fv2.to_string(index=False))

In [ ]:
def check_fig2a(t):
    t = t.set_index("level")
    c360 = t.pi_cath.get(360, np.nan)
    a570 = t.pi_anod.get(570, np.nan)
    out = dict(cath_360=c360, cath_360_ok=abs(c360 - 66) <= 6.6, anod_570=a570,
               anod_570_ok=(abs(a570 - (-83)) <= 8.3) if not np.isnan(a570) else None,
               cath_360_is_max=bool(c360 >= t.pi_cath.max()),
               max_cath=t.pi_cath.max(), max_cath_level=int(t.pi_cath.idxmax()),
               min_anod=t.pi_anod.min(), min_anod_level=int(t.pi_anod.idxmin()))
    hi = [L for L in (1000, 1200, 1400) if L in t.index]
    out["levels_1000_1400"] = len(hi)
    if hi and not np.isnan(a570):
        out["anod_not_more_neg"] = bool(all(t.pi_anod[L] >= a570 - 8.3 for L in (780, 1000, 1200, 1400) if L in t.index))
        out["cath_hi_below_360"] = bool(all(t.pi_cath[L] < c360 for L in hi))
    return out

rows = []
for bgname, background in [("b", True), ("a", False)]:
    t_all = show_levels("FV", ["pi_cath", "pi_anod"], background)
    for (mapping, reading), t in t_all.groupby(["mapping", "cells"]):
        rows.append(dict(bg=bgname, mapping=mapping, cells=reading, **check_fig2a(t)))
print(pd.DataFrame(rows).round(1).to_string(index=False))
allsets = units(D[D.dye == "FV"], "experiment").groupby("volt")[["pi_cath", "pi_anod"]].mean()
pe = D[D.dye == "FV"].groupby(["subject", "volt"])[["pi_cath", "pi_anod"]].mean().unstack("subject")
lv = pef.nearest_sets(D.volt, K["bottom"])
print("Per experiment, bottom mapping levels (mean of each experiment's cells):")
print(pe.loc[list(lv.values())].round(1).to_string())
print(f"\nOver all 19 sets (experiment means, background subtracted): largest cathodic {allsets.pi_cath.max():.1f}% at "
      f"{allsets.pi_cath.idxmax()} V, most negative anodic {allsets.pi_anod.min():.1f}% at {allsets.pi_anod.idxmin()} V")

**Result: not reproduced.** At 360 V/cm the per-image cathodic dF is 25.5 +- 2.7% under the bottom
mapping (experiment means; 26.3 for the cell nearest the centre, 27.7 over all cells), against
66 +- 6.6, and 20.1 to 24.1% under the top mapping. The largest cathodic mean of any set is 36.1%
(120 V), and no single experiment exceeds 41.8% at the four levels of the bottom mapping.
The anodic value at 570 V/cm exists only under the top mapping, where it is -33.7 to
-38.6%, against -83 +- 8.3; the most negative anodic mean of any set is -49.5%. Without
background subtraction the values are about half as large. Under the top mapping the cathodic
response falls from 35.9% at 570 V/cm to 21.2% at 1400 V/cm, and the anodic one keeps growing
from -35.8% to -49.4%.

## fig2a-fluovolt-stable

> FluoVolt measurements were stable without pulses (no significant bleaching or
> photosensitization), and the response within each set stayed stable through the 60 pulses,
> even at 1000 V/cm and above.

Per set and pole, the start and end of the 40 baseline images, and of the 60 pulse images,
compared by a 2-tailed paired t-test over units at p < 0.05. Start and end are (a) the first
and the last lit image, (b) the first 10 and the last 10. The sets are those nearest the
paper's levels (main), with all 19 sets as a variant.

In [ ]:
def start_end(frames, lo, hi, mode):
    seg = frames[lo:hi]
    ok = np.nonzero(~np.isnan(seg))[0]
    if mode == "a":
        return seg[ok[0]], seg[ok[-1]]
    return np.nanmean(seg[:10]), np.nanmean(seg[-10:])

def stability(dye, phase, mapping, reading, mode, sets_sel="levels", background=True):
    df = (D if background else D_raw)
    df = df[df.dye == dye].copy()
    lo, hi = (0, pef.N_BASE) if phase == "baseline" else (pef.N_BASE, 100)
    for pole in ["cath", "anod"]:
        se = np.array([start_end(f, lo, hi, mode) for f in df[f"frames_{pole}"]])
        df[f"s_{pole}"], df[f"e_{pole}"] = se[:, 0], se[:, 1]
    u = units(df, reading)
    if sets_sel == "levels":
        lv = pef.nearest_sets(df.volt, K[mapping])
        pick = {v: L for L, v in lv.items()}
    else:
        pick = {v: round(v * K[mapping]) for v in sorted(df.volt.unique())}
    rows = []
    for v, L in pick.items():
        g = u[u.volt == v]
        for pole in ["cath", "anod"]:
            t, p = stats.ttest_rel(g[f"s_{pole}"], g[f"e_{pole}"])
            rows.append(dict(level=L, volt=v, pole=pole, n=len(g), start=g[f"s_{pole}"].mean(), end=g[f"e_{pole}"].mean(),
                             change=(g[f"e_{pole}"] - g[f"s_{pole}"]).mean(), p=p))
    return pd.DataFrame(rows)

summ = []
for mapping in ["bottom", "top"]:
    for reading in READINGS:
        for mode in ["a", "b"]:
            for sets_sel in ["levels", "all"]:
                b = stability("FV", "baseline", mapping, reading, mode, sets_sel)
                q = stability("FV", "pulse", mapping, reading, mode, sets_sel)
                qh = q[q.level >= 1000]
                summ.append(dict(mapping=mapping, cells=reading, start_end=mode, sets=sets_sel,
                                 baseline_tests=len(b), baseline_sig=int((b.p < 0.05).sum()),
                                 baseline_max_abs_change=b.change.abs().max(),
                                 pulse_1000plus_tests=len(qh), pulse_1000plus_sig=int((qh.p < 0.05).sum()),
                                 pulse_1000plus_max_abs_change=qh.change.abs().max() if len(qh) else np.nan))
sm = pd.DataFrame(summ)
print(sm.round(2).to_string(index=False))

In [ ]:
print("Main reading (bottom mapping, experiment means, first 10 against last 10), baseline per set and pole:")
print(stability("FV", "baseline", "bottom", "experiment", "b").round(3).to_string(index=False))
print("\nTop mapping, experiment means, first 10 against last 10, pulse images at 1000 to 1400 V/cm:")
q = stability("FV", "pulse", "top", "experiment", "b")
print(q[q.level >= 1000].round(3).to_string(index=False))

**Result: not reproduced.** In the baseline images (bottom mapping, experiment means, sets at the paper's
levels), the cathodic pole at 150 V/cm (90 V) changes by -2.65% from the first 10 to the last 10
images (2-tailed paired t-test p = 0.010, n = 5). With the first and the last image, 1 of 8
baseline tests is significant as well. The other baseline changes lie between -2.2% and +0.02%.
With the cell nearest the centre and 10-image windows, none of 8 differ; over all 19 sets, 4 to
9 of 38 tests differ depending on the reading. The pulse sets at 1000 to 1400 V/cm exist only
under the top mapping. There, with experiment means, none of the 6 tests differs (changes -0.8
to -2.0%, p 0.09 to 0.54).

## fig4a-fluovolt-limits

> FluoVolt measured and calculated IMP matched only at 43 and 150 V/cm; with stronger 1-us
> pulses the pole dF peaked at 71 +- 10% (cathodic) and -83.5 +- 5% (anodic), about +260 and
> -330 mV.

Collapsed pole dF per cell and level, mean across units; the maximum is the largest mean |dF|
over the levels at each pole. Measured IMP (dF / 26 x 100 mV) against the cell's pole IMPC by
2-tailed paired and unpaired t-tests at p < 0.05.

In [ ]:
def imp_vs_impc(dye, mapping, reading, background=True):
    df = (D if background else D_raw)
    df = df[df.dye == dye].copy()
    k = K[mapping]
    df["imp_cath"] = df.dF_cath / pef.SENS[dye] * 100
    df["imp_anod"] = df.dF_anod / pef.SENS[dye] * 100
    df["impc_cath"] = pef.impc_mV(df.volt * k, df.R_area_um, 0)
    df["impc_anod"] = -df.impc_cath
    u = units(df, reading)
    lv = pef.nearest_sets(df.volt, k)
    rows = []
    for L, v in lv.items():
        g = u[u.volt == v]
        for pole in ["cath", "anod"]:
            meas, calc = g[f"imp_{pole}"], g[f"impc_{pole}"]
            rows.append(dict(level=L, volt=v, pole=pole, n=len(g), dF=g[f"dF_{pole}"].mean(), dF_se=g[f"dF_{pole}"].sem(),
                             imp=meas.mean(), impc=calc.mean(), ratio=meas.mean() / calc.mean(),
                             p_paired=stats.ttest_rel(meas, calc).pvalue, p_unpaired=stats.ttest_ind(meas, calc).pvalue))
    return pd.DataFrame(rows)

def limits_check(t, dye, exp_c, tol_c, exp_a, tol_a):
    c = t[t.pole == "cath"].set_index("level")
    a = t[t.pole == "anod"].set_index("level")
    return dict(max_cath=c.dF.max(), at=int(c.dF.idxmax()), max_cath_ok=abs(c.dF.max() - exp_c) <= tol_c,
                min_anod=a.dF.min(), at_a=int(a.dF.idxmin()), min_anod_ok=abs(a.dF.min() - exp_a) <= tol_a)

rows = []
for mapping in ["bottom", "top"]:
    for reading in READINGS:
        t = imp_vs_impc("FV", mapping, reading)
        r = dict(mapping=mapping, cells=reading, **limits_check(t, "FV", 71, 10, -83.5, 5))
        for test in ["p_paired", "p_unpaired"]:
            low = t[t.level.isin([43, 150])]
            high = t[t.level >= 230]
            r[f"{test}_no_diff_43_150"] = bool((low[test] >= 0.05).all()) if len(low) else None
            below = high.assign(b=(high[test] < 0.05) & (high.imp.abs() < high.impc.abs()))
            r[f"{test}_below_each_230plus"] = bool(below.groupby("level").b.any().all())
        rows.append(r)
print(pd.DataFrame(rows).round(1).to_string(index=False))
print("\nPer level, experiment means:")
for mapping in ["bottom", "top"]:
    print(mapping)
    print(imp_vs_impc("FV", mapping, "experiment").round(3).to_string(index=False))

**Result: not reproduced.** The largest mean cathodic dF is 35.3% under the bottom mapping (at 230 V/cm;
31.7 and 35.8 under the other cell readings), against 71 +- 10, and the most negative anodic
value is -49.5% (360 V/cm), against -83.5 +- 5. Under the top mapping they are 34.4 to 35.9% and
-49.5 to -51.1%. At 43 and 150 V/cm the measured cathodic IMP is below IMPC (bottom, experiment
means: ratios 0.34 and 0.80, paired p < 0.001 and p = 0.021), so the match there also fails.
Measured IMP is below IMPC at each level from 230 V/cm (ratios 0.27 to 0.67).

## fig4a-fluovolt-cathodic-decline

> FluoVolt cathodic polarization fell from 71% at 360 V/cm to 45 to 50% at 1000 to 1400 V/cm.

Collapsed cathodic pole dF, mean across units, at 360 V/cm (71 +- 10%) and at each of 1000, 1200
and 1400 V/cm (40.5 to 55%).

In [ ]:
rows = []
for mapping in ["bottom", "top"]:
    for reading in READINGS:
        t = imp_vs_impc("FV", mapping, reading)
        c = t[t.pole == "cath"].set_index("level").dF
        r = dict(mapping=mapping, cells=reading, cath_360=c.get(360, np.nan), ok_360=abs(c.get(360, np.nan) - 71) <= 10)
        for L in (1000, 1200, 1400):
            r[f"cath_{L}"] = c.get(L, np.nan)
            r[f"ok_{L}"] = (40.5 <= c[L] <= 55) if L in c.index else None
        rows.append(r)
print(pd.DataFrame(rows).round(1).to_string(index=False))

**Result: not reproduced.** The collapsed cathodic dF at 360 V/cm is 25.5% under the bottom mapping
(26.3 and 27.7 under the other cell readings) and 20.1 to 24.1% under the top mapping, against
71 +- 10. Under the top mapping the values at 1000, 1200 and 1400 V/cm are 28.9, 25.5 and 21.2%
(experiment means), below the range 40.5 to 55%. Under the bottom mapping there are no sets
above 430 V/cm.

## Di-8-ANEPPS claims at 570 V/cm and above (blocked under the adopted mapping)

Under the adopted `bottom` mapping the sets reach 430 V/cm, and the claims below need sets at 570
to 1400 V/cm. They are blocked. Their criteria are run here under the `top` mapping, as the
variant the assumption sets aside; there 43 V/cm has no set, and 1200 V/cm is the 200 V set
(1120 V/cm). Under `bottom`, the values at the levels present are shown.

### fig2b-di8-climbs-to-570

> Di-8-ANEPPS dF kept increasing with field strength up to 570 V/cm at both poles.

Per-image pole dF averaged over the 60 pulse images, per cell, then across units; |dF| must
increase at every step from 43 to 570 V/cm.

In [ ]:
di2 = show_levels("Di8", ["pi_cath", "pi_anod"])
rows = []
for (mapping, reading), t in di2.groupby(["mapping", "cells"]):
    t = t.set_index("level").sort_index()
    t = t[t.index <= 570]
    steps = {pole: np.diff(np.abs(t[f"pi_{pole}"].values)) for pole in ["cath", "anod"]}
    rows.append(dict(mapping=mapping, cells=reading, levels=",".join(map(str, t.index)), has_570=570 in t.index,
                     cath=" ".join(f"{x:.1f}" for x in t.pi_cath), anod=" ".join(f"{x:.1f}" for x in t.pi_anod),
                     every_step_up=bool(all((s > 0).all() for s in steps.values()))))
print(pd.DataFrame(rows).to_string(index=False))

**Result: blocked (data missing under the adopted mapping).** Under the bottom mapping the four levels
present (43 to 360 V/cm) rise at every step at both poles with experiment means (cathodic -1.6,
11.4, 18.1, 22.5%; anodic -6.2, -15.6, -20.0, -27.8%), but there is no 570 V/cm set. Under the
top mapping (no 43 V/cm set) both poles rise at every step from 150 to 570 V/cm with experiment
means and with all cells, but not with the cell nearest the centre.

### fig2b-di8-decline-within-set

> With Di-8-ANEPPS, responses to pulses of 780 V/cm and above decreased gradually within each
> set of 60 images.

Mean |dF| at the start and end of the pulse images, mean across units; start and end (a) first
and last image, (b) first 10 and last 10; poles (c) cathodic, (d) both. The 2-tailed paired
t-test per set.

In [ ]:
rows = []
for reading in READINGS:
    for mode in ["a", "b"]:
        q = stability("Di8", "pulse", "top", reading, mode)
        q = q[q.level >= 780].copy()
        q["abs_start"], q["abs_end"] = q.start.abs(), q.end.abs()
        for L, g in q.groupby("level"):
            for _, r in g.iterrows():
                rows.append(dict(cells=reading, start_end=mode, level=L, volt=r.volt, pole=r.pole,
                                 abs_start=r.abs_start, abs_end=r.abs_end, end_below_start=r.abs_end < r.abs_start, p=r.p))
dd = pd.DataFrame(rows)
print("Top mapping:")
print(dd.round(3).to_string(index=False))
print("\nEnd below start in every set from 780 V/cm, by reading:")
print(dd.groupby(["cells", "start_end", "pole"]).end_below_start.all().unstack().to_string())

**Result: blocked (data missing under the adopted mapping).** Under the top mapping, the cathodic |dF|
at the end of the pulse images is below the start in every set from 780 V/cm under every cell
reading and both windows. With experiment means and 10-image windows it goes from 19.8 to 17.1,
22.2 to 19.7, 24.4 to 20.8 and 27.0 to 22.4% (p = 0.034, 0.069, 0.037, 0.142). The anodic pole
does not decline in most of these sets.

### fig4b-di8-limits

> With Di-8-ANEPPS and 1-us pulses, measured IMP fell below the calculated value at 570 V/cm,
> where dF peaked at 57 +- 14% (cathodic) and -55 +- 4% (anodic), over 600 mV.

In [ ]:
rows = []
for mapping in ["bottom", "top"]:
    for reading in READINGS:
        t = imp_vs_impc("Di8", mapping, reading)
        r = dict(mapping=mapping, cells=reading, **limits_check(t, "Di8", 57, 14, -55, 4))
        for test in ["p_paired", "p_unpaired"]:
            at570 = t[t.level == 570]
            r[f"{test}_below_at_570"] = bool(((at570[test] < 0.05) & (at570.imp.abs() < at570.impc.abs())).any()) if len(at570) else None
            low = t[t.level <= 360]
            r[f"{test}_not_below_43_360"] = bool(not ((low[test] < 0.05) & (low.imp.abs() < low.impc.abs())).any())
        rows.append(r)
print(pd.DataFrame(rows).round(1).to_string(index=False))
print("\nPer level, experiment means:")
for mapping in ["bottom", "top"]:
    print(mapping)
    print(imp_vs_impc("Di8", mapping, "experiment").round(3).to_string(index=False))

**Result: blocked (data missing under the adopted mapping).** Under the bottom mapping the largest mean
pole values are 22.5% and -27.8%, both at 360 V/cm, and there is no 570 V/cm set. Under the top
mapping Di-8-ANEPPS pole dF is largest at 1400 V/cm (24.3% and -29.2%), not at 570 V/cm (13.0%
and -16.7%), against 57 +- 14 and -55 +- 4. There, measured IMP is below IMPC at every level
(ratios 0.21 to 0.39 from 360 V/cm).

### fig4b-first-10-pulses

> Averaged over only the first 10 of the 60 pulses, Di-8-ANEPPS cathodic polarization reached
> 66 +- 15% (710 +- 200 mV), so the 600-mV maximum is not the dye's limit.

In [ ]:
rows = []
for mapping in ["bottom", "top"]:
    for reading in READINGS:
        t = show_levels("Di8", ["f10_cath", "dF_cath"], readings=[reading])
        t = t[t.mapping == mapping].set_index("level")
        rows.append(dict(mapping=mapping, cells=reading, max_first10=t.f10_cath.max(), at=int(t.f10_cath.idxmax()),
                         within_66_15=abs(t.f10_cath.max() - 66) <= 15, max_60=t.dF_cath.max(),
                         above_60_max=t.f10_cath.max() > t.dF_cath.max()))
print(pd.DataFrame(rows).round(1).to_string(index=False))

**Result: blocked (depends on fig4b-di8-limits).** Over the first 10 pulse images the largest mean
cathodic dF is 24.4% (bottom, 360 V/cm) and 27.0% (top, 1400 V/cm), against 66 +- 15. It is
above the 60-image maximum (22.5 and 24.3%) under every reading.

### fig4b-cathodic-asymmetry

> Above 780 V/cm the pole IMP fell progressively with field strength, more at the cathodic pole;
> at 1400 V/cm the cathodic pole no longer charged and the anodic pole fell to about -30% dF.

In [ ]:
rows = []
for reading in READINGS:
    u = units(D[D.dye == "Di8"], reading)
    lv = pef.nearest_sets(u.volt, K["top"])
    m = {L: u[u.volt == v] for L, v in lv.items()}
    c = {L: g.dF_cath.mean() for L, g in m.items()}
    a = {L: g.dF_anod.mean() for L, g in m.items()}
    dec = all(abs(c[L]) < abs(c[780]) for L in (1000, 1200, 1400)) and all(abs(a[L]) < abs(a[780]) for L in (1000, 1200, 1400))
    lost_c = 1 - abs(c[1400]) / abs(c[780])
    lost_a = 1 - abs(a[1400]) / abs(a[780])
    t1 = stats.ttest_1samp(m[1400].dF_cath, 0)
    rows.append(dict(cells=reading, cath=" ".join(f"{c[L]:.1f}" for L in (780, 1000, 1200, 1400)),
                     anod=" ".join(f"{a[L]:.1f}" for L in (780, 1000, 1200, 1400)), decreases=dec,
                     lost_cath=lost_c, lost_anod=lost_a, more_at_cath=lost_c > lost_a, cath_1400=c[1400],
                     p_cath_1400_vs_0=t1.pvalue, anod_1400=a[1400], anod_ok=abs(a[1400] + 30) <= 3))
print("Top mapping (levels 780, 1000, 1200, 1400 V/cm):")
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: blocked (data missing under the adopted mapping).** Under the top mapping both poles keep
growing from 780 to 1400 V/cm (cathodic 18.1, 20.4, 22.5, 24.3%; anodic -21.2, -25.7, -27.8,
-29.2%, experiment means), so the decline is absent. The cathodic value at 1400 V/cm differs from
zero (one-sample p < 0.001), and the anodic -29.2% is within -30 +- 3.

### fig4c-equator-follows-cosine

> Membrane regions nearer the equator were unaffected by pole electroporation and followed the
> cosine dependence of the Schwan equation up to 1000 V/cm.

Measured IMP (dF / 9.1%) against IMPC, averaged per unit within arcs beside the two equator
points (90 and 270 degrees from the cathode), (a) 15 and (b) 30 degrees wide, on the cathodic and
anodic side of each; 2-tailed paired t-test at 570, 780 and 1000 V/cm.

In [ ]:
def equator_arcs(width):
    th = np.arange(360)
    return {"90_cath_side": (th >= 90 - width) & (th < 90), "90_anod_side": (th > 90) & (th <= 90 + width),
            "270_anod_side": (th >= 270 - width) & (th < 270), "270_cath_side": (th > 270) & (th <= 270 + width)}

rows = []
for reading in READINGS:
    df = D[D.dye == "Di8"]
    lv = pef.nearest_sets(df.volt, K["top"])
    for width in (15, 30):
        for L in (570, 780, 1000):
            g = df[df.volt == lv[L]]
            for name, sel in equator_arcs(width).items():
                meas = np.array([p[sel].mean() for p in g.profile]) / pef.SENS["Di8"] * 100
                calc = np.array([pef.impc_mV(lv[L] * K["top"], r, np.arange(360))[sel].mean() for r in g.R_area_um])
                x = g.assign(meas=meas, calc=calc)
                x = units(x, reading) if reading != "experiment" else x.groupby("subject", as_index=False)[["meas", "calc"]].mean()
                p = stats.ttest_rel(x.meas, x.calc).pvalue
                rows.append(dict(cells=reading, width=width, level=L, arc=name, n=len(x), meas=x.meas.mean(), calc=x.calc.mean(),
                                 ratio=x.meas.mean() / x.calc.mean(), p=p, differs=p < 0.05))
eq = pd.DataFrame(rows)
print("Top mapping:")
print(eq.round(3).to_string(index=False))
print("\nArcs that differ, by reading:", eq.groupby(["cells", "width"]).differs.sum().to_dict())

**Result: blocked (data missing under the adopted mapping).** Under the top mapping, measured IMP in the
arcs beside the equator is below IMPC at 570, 780 and 1000 V/cm in most arcs (ratios -0.15 to
0.82). Of 12 arc-level tests per reading, 9 to 11 differ at p < 0.05 with 15-degree arcs and all 12
with 30-degree arcs.

### fig1-pole-dimming

> With Di-8-ANEPPS and 1-us pulses, polarization grows with field strength up to 570 V/cm; at
> 780 V/cm the cathodic pole stops increasing and dims, and at 1000 to 1400 V/cm the poles show
> reduced or absent polarization while regions nearer the equator stay polarized and may
> polarize more than the poles.

(1) The mean cathodic pole dF at 780 V/cm is not above that at 570 V/cm; (2) at 1400 V/cm the
largest mean dF of the cathode-facing hemisphere lies outside the +-5 degree polar arc.
Backgrounds (a) and (b).

In [ ]:
rows = []
for bgname, df0 in [("b", D), ("a", D_raw)]:
    for reading in READINGS:
        u = units(df0[df0.dye == "Di8"], reading)
        lv = pef.nearest_sets(u.volt, K["top"])
        c570 = u[u.volt == lv[570]].dF_cath.mean()
        c780 = u[u.volt == lv[780]].dF_cath.mean()
        g14 = units(df0[(df0.dye == "Di8") & (df0.volt == lv[1400])], "cells" if reading == "cells" else "centre" if reading == "centre" else "cells")
        if reading == "experiment":
            prof = np.mean([np.mean(np.stack(g.profile.values), 0) for _, g in g14.groupby("subject")], 0)
        else:
            prof = np.mean(np.stack(g14.profile.values), 0)
        hemi = np.r_[np.arange(270, 360), np.arange(0, 91)]
        peak = hemi[np.argmax(prof[hemi])]
        off = (peak + 180) % 360 - 180
        rows.append(dict(bg=bgname, cells=reading, cath_570=c570, cath_780=c780, cond1=c780 <= c570,
                         peak_deg_from_cathode=off, peak_dF=prof[peak], pole_dF=pef.arc(prof, 0), cond2=abs(off) > 5))
print("Top mapping:")
print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: blocked (data missing under the adopted mapping).** Under the top mapping the mean cathodic
dF at 780 V/cm (18.1%) is above that at 570 V/cm (13.0%), so condition (1) fails under every
reading. At 1400 V/cm the peak of the cathode-facing hemisphere lies 12 to 46 degrees from the
pole (condition 2 holds).

### fig7b-charging-during-pulse

> Normalized to 150 V/cm, pole polarization during the conditioning pulse fell 1.5 times below
> linear scaling at 560 V/cm and 2-fold at 760 V/cm, and further at stronger fields (p < 0.01 for
> every point from 560 V/cm).

Collapsed pole dF as % of the 150 V/cm value, against the line 100% x E / 150. Normalization
(a) per cell, (b) of the means; pole (c) each, (d) mean of the poles. One-sample t-test of the
per-unit values against the line.

In [ ]:
rows = []
for reading in READINGS:
    u = units(D[D.dye == "Di8"], reading)
    lv = pef.nearest_sets(u.volt, K["top"])
    base = u[u.volt == lv[150]].set_index("unit")
    for pole in ["cath", "anod", "mean"]:
        def val(g):
            return (g.dF_cath - g.dF_anod) / 2 if pole == "mean" else g[f"dF_{pole}"].abs()
        b = val(base)
        r = dict(cells=reading, pole=pole)
        okp = True
        for L in (570, 780, 1000, 1200, 1400):
            g = u[u.volt == lv[L]].set_index("unit")
            E = lv[L] * K["top"]
            line = 100 * E / (lv[150] * K["top"])
            pc = 100 * val(g) / b
            r[f"ratio_a_{L}"] = line / pc.mean()
            r[f"ratio_b_{L}"] = line / (100 * val(g).mean() / b.mean())
            p = stats.ttest_1samp(pc, line).pvalue
            r[f"p_{L}"] = p
            okp &= bool(p < 0.01 and pc.mean() < line)
        r["all_below_p01"] = okp
        rows.append(r)
f7 = pd.DataFrame(rows)
print("Top mapping (560 V/cm is the 100 V set, 760 V/cm the 140 V set at 784 V/cm):")
print(f7.round(3).to_string(index=False))

**Result: blocked (data missing under the adopted mapping).** Under the top mapping, normalized to the
150 V/cm set (25 V, where bleaching makes the cathodic dF negative in some cells), the
linear-over-measured ratio with experiment means is 1.46 at 560 V/cm and 1.62 at 760 V/cm at the
anodic pole (expected 1.5 +- 0.15 and 2 +- 0.2), and below 1 at the cathodic pole. Every anodic
datapoint is below the line at p < 0.01 with experiment means, where the ratios at 1000 to 1400
V/cm are 1.72 to 2.08; no other reading has every datapoint below the line at p < 0.01.

## fig5b-conductivity-calibration

> By Eq. (2), the effective membrane conductivity barely affects charging while it stays far
> below cytosol and medium conductivities; a 10-fold rise reduces IMP by about 2% and a 100-fold
> rise by 12%.

Eq. (2) with the constants of Methods 2.3 (lambda_i 0.4 S/m, lambda_o 1.64 S/m, R 8 um, d 5 nm,
intact lambda_m 3e-7 S/m); no recordings. Reduction read as (a) 1 - f/1.5 and (b) 1 - f(lambda_m)
/ f(3e-7 S/m).

In [ ]:
f0 = pef.f_factor(3e-7)
print(f"f at the intact 3e-7 S/m: {f0:.5f} (1.5 for a non-conducting membrane)")
rows = []
for fold in [1, 3, 10, 30, 100, 300, 1000]:
    f = pef.f_factor(3e-7 * fold)
    rows.append(dict(fold=fold, lambda_m=3e-7 * fold, f=f, reduction_a=100 * (1 - f / 1.5), reduction_b=100 * (1 - f / f0)))
t5 = pd.DataFrame(rows)
print(t5.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
for fold, e, tol in [(10, 2, 0.2), (100, 12, 1.2)]:
    r = t5.set_index("fold").loc[fold]
    print(f"{fold}-fold: (a) {r.reduction_a:.2f}%, (b) {r.reduction_b:.2f}%; expected {e} +- {tol}%;"
          f" within: (a) {abs(r.reduction_a - e) <= tol}, (b) {abs(r.reduction_b - e) <= tol}")
fold_for_2 = [fo for fo in np.logspace(0, 3, 3001) if 100 * (1 - pef.f_factor(3e-7 * fo) / 1.5) >= 2][0]
print(f"A 2% reduction, reading (a), needs a {fold_for_2:.1f}-fold rise")

**Result: partially reproduced.** Eq. (2) with the constants of Methods 2.3 gives f = 1.498 for the
intact membrane. A 10-fold rise in conductivity reduces IMP by 1.33% under reading (a) and 1.20%
under (b), against 2 +- 0.2; it takes a 15.2-fold rise to reach 2%. A 100-fold rise reduces IMP
by 11.87% and 11.75%, within 12 +- 1.2. The claim's qualitative point holds: the reduction stays
small until the conductivity rises more than 30-fold (3.9% at 30-fold).

## Summary

| Claim | Outcome |
|---|---|
| fig3-dye-sensitivity | partially reproduced |
| fig3-fluovolt-nonlinear | not reproduced |
| fig2a-fluovolt-saturation | not reproduced |
| fig2a-fluovolt-stable | not reproduced |
| fig4a-fluovolt-limits | not reproduced |
| fig4a-fluovolt-cathodic-decline | not reproduced |
| fig5b-conductivity-calibration | partially reproduced |
| fig1-pole-dimming | blocked: data missing (no sets above 430 V/cm under the adopted mapping) |
| fig2b-di8-climbs-to-570 | blocked: data missing |
| fig2b-di8-decline-within-set | blocked: data missing |
| fig4b-di8-limits | blocked: data missing |
| fig4b-first-10-pulses | blocked: data missing |
| fig4b-cathodic-asymmetry | blocked: data missing |
| fig4c-equator-follows-cosine | blocked: data missing |
| fig7b-charging-during-pulse | blocked: data missing |
| fig2cd-stable-before-electroporation, fig2c-1000-within-set, fig2d-lasting-reduction | blocked: data missing |
| fig4d-short-pulses-same-limit | blocked: data missing |
| fig5a-gd-extends-anodic, fig5a-gd-linear, fig5a-gd-cathodic, fig5a-gd-differences, fig5c-gd-reduces-conductivity | blocked: data missing |
| fig6a-ca-single-pulse, fig6b-ca-train-responders, fig6b-ca-650 | blocked: data missing |
| fig7ab-test-pulse-recovery, fig7c-fast-conductivity-drop, fig7c-full-reseal-mild | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The files give the pulse in volts. The factor to V/cm is taken as 43 V/cm at 25 V (1.72 V/cm
  per V), an assumption supported by the low-field response of both dyes under the paper's
  calibration; the other endpoint mapping (5.6 V/cm per V) is shown as a variant. Sets are
  named by the paper's level nearest them.
- The paper analysed one cell per experiment; the files show 3 to 6 cells per field and do not
  say which. Experiment means, the cell nearest the image centre, and all cells are each run.
- Membrane traces are found automatically, not drawn by hand; dark frames are left out, so
  most baselines have 39 images; the pixel size (0.267 um) is assumed from the camera and
  objective; tau is from reference [20].
- The mean of per-image pole dF and the collapsed-image pole dF are the same number here,
  because both use the same baseline mean as F0, so the Fig. 2 and Fig. 4 measures coincide.

What was found about the dataset:

- It holds ten 19-set escalations of 1-us pulses (25 to 250 V), five per dye, from one day. It
  holds none of the paper's alternating, 0.3-us, Gd3+, Ca2+ or two-pulse experiments. Its
  voltages span a factor of 10, so it cannot hold the paper's 43 to 1400 V/cm series; under the
  adopted mapping it covers 43 to 430 V/cm.
- The files state an electrode distance of 1.00 mm; the paper states 1.6 mm. They give no field
  strength, field direction, pixel size or cell identity. The field direction is clear from the
  data (cathode-facing side towards the bottom of the image in every experiment).
- Frame 0 is dark in 175 of 190 sets, and 51 other frames are dark, which the files do not
  mention.
- No recording is deposited twice.